<div id="singlestore-header" style="display: flex; background-color: rgba(209, 153, 255, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/vector-circle.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Build Modern AI and Search Applications</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>Features marked <b>NEW</b> need SingleStore 10 or later. The notebook checks what your workspace supports and skips or falls back with a clear message, so it runs end to end on any Standard or Premium workspace.</p>
    </div>
</div>

The goal of this notebook is to demonstrate **F16 vectors, full-text search, index merging, and improved full-text index manageability** by building a real-time ad search app on SingleStore's Real-Time Digital Marketing dataset.

**The story:** you're the only developer at AdVenture Inc., an ad startup. Your CMO, Morgan, sends requests to `#war-room`, and each one unlocks a feature:

| Chapter | Morgan asks for | You use |
| --- | --- | --- |
| 1 | Real ads, not receipts | Pipelines + AI Functions |
| 2 | Keyword search | Full-text search, <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> `SHOW FULLTEXT INDEX STATUS` |
| 3 | "run" should find "running" | <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> `ANALYZE FULLTEXT`, analyzer swap |
| 4 | Ads that *feel* like a rainy Sunday | <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> F16 vectors |
| 5 | Both at once | Hybrid search |
| 6 | Stay fast on Super Bowl night | Vector index merging, <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> full-text index merger |

## Prerequisite: generate the Digital Marketing dataset

The Digital Marketing dataset can be generated using the website here: [digital-marketing.labs.singlestore.com](https://digital-marketing.labs.singlestore.com/)

1. Make sure your Standard or Premium workspace is running.
2. Open the website and fill in the connection form: your workspace host with `https://` in front (find it under **Connect** in the portal), username `admin`, your workspace password, and database name `martech`. Click **Connect**.
3. Click **Setup Database**. It creates the `martech` database and starts streaming shopper locations and purchases into it. Keep the **Simulator** switch on.
4. Under **Locations**, tick the cities you want. Each city adds ad campaigns, so more cities give the notebook more ads to search.
5. Wait until **Ad Campaigns** shows a number above 0, then come back here.

If the connection fails, check that your workspace's firewall allows your IP address.

The notebook only reads from `martech`. If you skip this step, it loads the same data from the dataset's public S3 bucket instead.

## Chapter 0: Setup

Start with the database drop-down at the top of the notebook **empty**. The setup cell creates a database called `adventure_inc` for everything this notebook builds.

In [1]:
%pip install singlestoredb pandas --quiet

The next cell connects, creates `adventure_inc`, then checks which features your workspace supports. If AI Functions aren't enabled, it loads a free open-source embedding model instead (about a minute).

In [2]:
import json, random, re, statistics, time   # repeated here so this cell works on its own
import pandas as pd
import singlestoredb as s2

pd.set_option('display.max_colwidth', 120)
random.seed(42)

DB_NAME = 'adventure_inc'

USE_DB = None   # set below once we know which database to use

def connect():
    """Open a connection (again) and select the notebook's database."""
    global conn
    conn = s2.connect(database=USE_DB) if USE_DB else s2.connect()
    conn.autocommit(True)
    if USE_DB:
        with conn.cursor() as cur:
            cur.execute(f'USE {USE_DB}')

def _query(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description:
            cols = [d[0] for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)
    return None

def run_sql(query, params=None):
    """Run a query and return a DataFrame (or None).

    Handles two temporary conditions for you:
    - an idle connection that was dropped: reconnects and retries once
    - the Full Text Service still starting up on first use: waits and retries
    """
    for attempt in range(13):
        try:
            return _query(query, params)
        except Exception as e:
            msg = str(e)
            if getattr(e, 'errno', None) in (2006, 2013) or 'Lost connection' in msg or 'gone away' in msg:
                print('(Connection was dropped, reconnecting...)')
                connect()
            elif 'Full Text Service' in msg and attempt < 12:
                if attempt == 0:
                    print('(Full Text Service is starting up, retrying in a few seconds...)')
                time.sleep(5)
            else:
                raise

connect()

def try_sql(query, params=None, label='This step'):
    """Like run_sql(), but explains and continues if the workspace doesn't support something."""
    try:
        return run_sql(query, params)
    except Exception as e:
        print(f'{label} was skipped: {e}')
        return None

shared = run_sql("SHOW VARIABLES LIKE 'is_shared_tier'")
IS_SHARED = shared is not None and len(shared) > 0 and shared.iloc[0, 1] == 'ON'
if not IS_SHARED:
    run_sql(f'CREATE DATABASE IF NOT EXISTS {DB_NAME}')
    USE_DB = DB_NAME
    run_sql(f'USE {DB_NAME}')

print('Connected to database:', run_sql('SELECT DATABASE() AS db').iloc[0, 0])
print('SingleStore version:  ', run_sql('SELECT @@memsql_version AS v').iloc[0, 0])

# ---- Feature check ----
import logging, os, warnings
os.environ['HF_HUB_VERBOSITY'] = 'error'              # quiet the model download messages
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
for name in ['httpx', 'huggingface_hub', 'huggingface_hub.utils._http', 'sentence_transformers']:
    logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*unauthenticated requests.*')

# 1. F16 vectors (half precision)
try:
    run_sql("SELECT '[0.5, -0.25]' :> VECTOR(2, F16) AS v")
    VEC_ELEM = 'F16'
    print('F16 vectors: supported')
except Exception as e:
    VEC_ELEM = 'F32'
    print('F16 vectors: not available on this workspace, falling back to F32 ->', e)

# 2. AI Functions for embeddings
local_model = None
try:
    probe = run_sql("SELECT LENGTH(cluster.EMBED_TEXT('hello from AdVenture Inc.')) AS n_bytes")
    DIM = int(probe.iloc[0, 0]) // 4   # EMBED_TEXT returns packed 32-bit floats
    run_sql(f"SELECT cluster.EMBED_TEXT('check') :> VECTOR({DIM}) :> VECTOR({DIM}, {VEC_ELEM}) AS v")
    EMBED_MODE = 'ai'
    print(f'AI Functions: available, embeddings have {DIM} dimensions')
except Exception as e:
    EMBED_MODE = 'local'
    print('AI Functions: not available, using a free open-source model instead ->', e)
    print('Downloading the model from Hugging Face (one time, about a minute)...')
    get_ipython().run_line_magic('pip', 'install sentence-transformers --quiet')
    from sentence_transformers import SentenceTransformer
    local_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    DIM = (getattr(local_model, 'get_embedding_dimension', None)
           or local_model.get_sentence_embedding_dimension)()

VEC_TYPE = f'VECTOR({DIM}, {VEC_ELEM})'
print('Embedding column type:', VEC_TYPE, '| embeddings made by:', EMBED_MODE)

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Now select <tt>adventure_inc</tt> from the database drop-down at the top of this notebook (refresh the list if it isn't there yet). This keeps every later cell, including the full-text steps in Chapter 2, connected to the right database.</p>
    </div>
</div>

## Chapter 1: Boring offers in, real ads out

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">8:02 AM</span> &nbsp;<b>Morgan (CMO):</b> Our offers say <i>"37% off at Wordtune"</i>. That's a receipt, not an ad. Make them real ads, searchable by lunch.
</div>

Each offer gets a headline and ad copy based on its advertiser's category, plus a vector from `cluster.EMBED_TEXT`, all in the same SQL step as the data loads. First, two lookup tables: the dataset's vendor list and a "vibe book" of copy per category.

In [3]:
vendors = pd.read_json('https://raw.githubusercontent.com/singlestore-labs/demo-realtime-digital-marketing/main/data/vendors.json')
vendors['domain'] = (vendors['vendor'].str.lower() + '.' + vendors['tld'])
vendors = vendors.drop_duplicates('domain')

run_sql('DROP TABLE IF EXISTS vendors')
run_sql('''
    CREATE ROWSTORE REFERENCE TABLE vendors (
        domain VARCHAR(200) PRIMARY KEY,
        vendor VARCHAR(100),
        category VARCHAR(50),
        popularity DOUBLE
    )
''')
with conn.cursor() as cur:
    cur.executemany(
        'INSERT INTO vendors VALUES (%s, %s, %s, %s)',
        vendors[['domain', 'vendor', 'category', 'popularity']].values.tolist(),
    )
run_sql('SELECT category, COUNT(*) AS vendors FROM vendors GROUP BY category ORDER BY vendors DESC LIMIT 8')

In [4]:
# The copywriter's vibe book: three hooks, a story and a call to action per category.
VIBES = {
    'books':       ('Plot twist: you finally read it', 'Your reading nook called', 'Chapter one starts tonight',
                    'Rain on the window, tea in hand, a stack of new stories and nowhere to be on a slow Sunday.', 'Curl up and start reading.'),
    'games':       ('Press start on the weekend', 'Couch co-op, snacks ready', 'One more level, we promise',
                    'Stay-in weekends, board game nights with friends and cozy gaming marathons when the weather turns grey.', 'Grab a controller.'),
    'movies':      ('Popcorn is mandatory', 'Your movie night just got an upgrade', 'Roll credits on boredom',
                    'Blanket forts, film marathons and date nights on the sofa while it pours outside.', 'Hit play.'),
    'music':       ('Turn it up', 'Your road trip needs a soundtrack', 'Headphones on, world off',
                    'Wireless headphones, vinyl crates and playlists for long drives, workouts and quiet rainy afternoons.', 'Find your sound.'),
    'electronics': ('Upgrade everything', 'Wireless and fearless', 'Charged up and ready',
                    'Wireless headphones, smart speakers and gadgets that make your home, commute and workouts easier.', 'Plug into the future.'),
    'computers':   ('Built for deadline day', 'Your laptop deserves better', 'Faster than your group chat',
                    'Laptops, monitors and keyboards for working from home, studying late and gaming after hours.', 'Power up your desk.'),
    'sports':      ('Game day ready', 'Run further, recover faster', 'Your personal best is waiting',
                    'Running shoes, team jerseys and gear for runners, weekend leagues and big game watch parties.', 'Get in the game.'),
    'outdoors':    ('Adventure is calling', 'Tents up, phones down', 'Trail mix not included',
                    'Camping gear, hiking boots and everything for a road trip into the mountains or a lazy lake weekend.', 'Head outside.'),
    'shoes':       ('Step it up', 'Blisters are so last season', 'Run the city in comfort',
                    'Running shoes, sneakers and boots made for long walks, city commutes and marathon training.', 'Find your fit.'),
    'clothing':    ('New season, new you', 'Your closet called, it is bored', 'Layer up in style',
                    'Cozy sweaters for rainy days, outfits for date night and everyday basics that just work.', 'Dress the part.'),
    'beauty':      ('Glow like you mean it', 'Self-care Sunday, sorted', 'Your skin says thank you',
                    'Skincare, candles and bath bombs for a relaxing spa day at home or getting ready for a night out.', 'Treat yourself.'),
    'jewelry':     ('Say it with sparkle', 'Anniversary panic? Solved', 'Shine on',
                    'Rings, necklaces and gifts for anniversaries, date nights, proposals and holiday surprises.', 'Make it unforgettable.'),
    'health':      ('Feel good from the inside', 'Wellness, minus the lecture', 'Your body will notice',
                    'Vitamins, yoga mats and sleep aids for calm mornings, healthy habits and a fresh start.', 'Start feeling better.'),
    'home':        ('Make your place a sanctuary', 'Cozy mode: activated', 'Home is where the throw pillows are',
                    'Blankets, candles and furniture that turn a grey rainy day into the coziest day of the year.', 'Nest a little.'),
    'garden':      ('Grow something good', 'Green thumbs welcome', 'Your balcony, but lush',
                    'Seeds, planters and tools for spring planting, herb gardens and slow weekends outdoors.', 'Get your hands dirty.'),
    'grocery':     ('Dinner, solved', 'Snack attack approved', 'Fresh from the market',
                    'Fresh groceries, comfort food ingredients and snacks for movie nights and busy weeknights.', 'Fill your cart.'),
    'kids':        ('Keep the little ones busy', 'Rainy day rescue kit', 'Fun for the whole family',
                    'Crafts, puzzles and activities for rainy days inside, school breaks and family weekends.', 'Make some memories.'),
    'toys':        ('Playtime, upgraded', 'The favorite toy is here', 'Bored? Not anymore',
                    'Building sets, puzzles and games for birthdays, holidays and rainy afternoons at home.', 'Let the fun begin.'),
    'baby':        ('Tiny humans, big deals', 'Sleep more (maybe)', 'Everything for the new arrival',
                    'Soft blankets, strollers and nursery essentials for new parents and baby showers.', 'Welcome the little one.'),
    'automotive':  ('Road trip ready', 'Your car deserves a spa day', 'Built for the long haul',
                    'Car care, tires and accessories for road trips, daily commutes and winter driving.', 'Hit the road.'),
    'tools':       ('Weekend project? Nailed it', 'Fix it yourself', 'Measure twice, save once',
                    'Drills, toolkits and supplies for DIY weekends, home repairs and garage projects.', 'Build something.'),
    'industrial':  ('Heavy duty, light price', 'Built for the job site', 'Tough gear for tough days',
                    'Safety gear, equipment and supplies for warehouses, workshops and serious builders.', 'Get to work.'),
    'general':     ('Deals worth stopping for', 'You will want to see this', 'Something good nearby',
                    'Everyday deals from local shops around the city.', 'Take a look.'),
}

run_sql('DROP TABLE IF EXISTS category_vibes')
run_sql('''
    CREATE ROWSTORE REFERENCE TABLE category_vibes (
        category VARCHAR(50) PRIMARY KEY,
        hook1 TEXT, hook2 TEXT, hook3 TEXT, story TEXT, cta TEXT
    )
''')
with conn.cursor() as cur:
    cur.executemany('INSERT INTO category_vibes VALUES (%s, %s, %s, %s, %s, %s)',
                    [(k, *v) for k, v in VIBES.items()])
print(f'{len(VIBES)} categories in the vibe book')

Now the tables. `ad_creatives` holds the ads and their vectors, and has no full-text index yet: you'll add one in Chapter 2 without rebuilding anything.

In [5]:
if 'VEC_TYPE' not in globals():
    raise RuntimeError('Run the setup cell in Chapter 0 first (it ends by printing "Embedding column type").')

MAX_CREATIVES = 3000   # caps AI Function calls per batch, keep it modest on a trial
SCALE = '100k-4p'      # dataset size in the bucket: 1k-2p, 10k-2p, 100k-4p, 1m-16p, ... 10m-80p

try_sql('DROP PIPELINE IF EXISTS purchases_pipeline', label='Dropping an old pipeline')
for t in ['purchases', 'offers', 'ad_creatives', 'creatives_staging']:
    run_sql(f'DROP TABLE IF EXISTS {t}')

run_sql('''
    CREATE TABLE purchases (
        subscriber_id BIGINT,
        vendor VARCHAR(100),
        ts DATETIME(6) DEFAULT NOW(6),
        SORT KEY (ts),
        KEY (vendor) USING HASH
    )
''')

run_sql('''
    CREATE ROWSTORE TABLE offers (
        offer_id BIGINT PRIMARY KEY,
        customer VARCHAR(100),
        enabled BOOL,
        notification_zone TEXT,
        segment_ids JSON,
        notification_content TEXT,
        notification_target VARCHAR(200),
        maximum_bid_cents BIGINT
    )
''')

run_sql('''
    CREATE ROWSTORE TABLE creatives_staging (
        offer_id BIGINT PRIMARY KEY,
        customer VARCHAR(100), category VARCHAR(50), domain VARCHAR(200),
        headline TEXT, body TEXT, max_bid_cents BIGINT
    )
''')

run_sql(f'''
    CREATE TABLE ad_creatives (
        offer_id BIGINT NOT NULL,
        customer VARCHAR(100),
        category VARCHAR(50),
        domain VARCHAR(200),
        headline TEXT,
        body TEXT,
        max_bid_cents BIGINT,
        embedding {VEC_TYPE},
        created_at DATETIME(6) DEFAULT NOW(6),
        SORT KEY (created_at),
        SHARD KEY (offer_id)
    )
''')
print('Tables ready.')

`build_creatives` writes the copy into a staging table, then moves it into `ad_creatives` while calling `cluster.EMBED_TEXT` in the same `INSERT ... SELECT`, so only new rows are ever embedded.

In [6]:
# Embed the ad's meaning, not the made-up brand names: strip the vendor name and domain first.
embed_text_sql = "REPLACE(REPLACE(CONCAT(s.headline, '. ', s.body), s.domain, ''), s.customer, '')"
embed_expr = (f"cluster.EMBED_TEXT({embed_text_sql}) :> VECTOR({DIM}) :> {VEC_TYPE}"
              if EMBED_MODE == 'ai' else 'NULL')

run_sql(f'''
CREATE OR REPLACE PROCEDURE build_creatives() AS
BEGIN
    INSERT INTO creatives_staging
    SELECT o.offer_id,
           o.customer,
           COALESCE(v.category, 'general'),
           o.notification_target,
           CONCAT(o.notification_content, ': ', ELT(1 + MOD(o.offer_id, 3), cv.hook1, cv.hook2, cv.hook3)),
           CONCAT(cv.story, ' ', o.customer, ' (', o.notification_target, ') bids up to ',
                  o.maximum_bid_cents, ' cents to reach you. ', cv.cta),
           o.maximum_bid_cents
    FROM offers o
    LEFT JOIN vendors v ON v.domain = o.notification_target
    JOIN category_vibes cv ON cv.category = COALESCE(v.category, 'general')
    WHERE NOT EXISTS (SELECT 1 FROM ad_creatives a WHERE a.offer_id = o.offer_id)
    LIMIT {MAX_CREATIVES};

    INSERT INTO ad_creatives (offer_id, customer, category, domain, headline, body, max_bid_cents, embedding)
    SELECT s.offer_id, s.customer, s.category, s.domain, s.headline, s.body, s.max_bid_cents,
           {embed_expr}
    FROM creatives_staging s;

    DELETE FROM creatives_staging;
END
''')

run_sql('''
CREATE OR REPLACE PROCEDURE refresh_ads() AS
BEGIN
    -- One offer per advertiser domain, generated the same way as the original demo app.
    INSERT IGNORE INTO offers
    SELECT CRC32(v.domain),
           v.vendor,
           TRUE,
           NULL,
           '[]',
           CONCAT(10 + MOD(CRC32(v.domain), 41), '% off at ', v.vendor),
           v.domain,
           2 + MOD(CRC32(v.domain), 14)
    FROM vendors v
    WHERE v.vendor IN (SELECT DISTINCT vendor FROM purchases);

    CALL build_creatives();
END
''')

run_sql('''
CREATE OR REPLACE PROCEDURE ingest_purchases(batch QUERY(subscriber_id BIGINT, vendor VARCHAR(100))) AS
BEGIN
    INSERT INTO purchases (subscriber_id, vendor) SELECT subscriber_id, vendor FROM batch;
    CALL refresh_ads();
END
''')

def embed_pending():
    """Local-model fallback: fill any NULL embeddings from Python."""
    if EMBED_MODE != 'local':
        return
    todo = run_sql('SELECT offer_id, customer, domain, headline, body FROM ad_creatives WHERE embedding IS NULL')
    if todo is None or todo.empty:
        return
    texts = [(h + '. ' + b).replace(d, '').replace(c, '')      # same brand-name stripping as the SQL path
             for h, b, d, c in zip(todo['headline'], todo['body'], todo['domain'], todo['customer'])]
    vecs = local_model.encode(texts, normalize_embeddings=True)
    with conn.cursor() as cur:
        cur.executemany('UPDATE ad_creatives SET embedding = %s WHERE offer_id = %s',
                        [(json.dumps([float(x) for x in v]), int(i)) for v, i in zip(vecs, todo['offer_id'])])
    print(f'Embedded {len(todo)} ads locally.')

print('Procedures ready.')

Load the data: from the demo's `martech` database if it's there, otherwise with a pipeline that streams purchases from S3 and creates ads for new advertisers as they appear.

In [7]:
DEMO_DB = 'martech'   # the database created by the Real-Time Digital Marketing demo

def load_from_s3():
    BUCKET_URL = 'https://singlestore-realtime-digital-marketing.s3.amazonaws.com'

    try:
        run_sql(f'''
            CREATE OR REPLACE PIPELINE purchases_pipeline AS
            LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/{SCALE}/purchases.*'
            CREDENTIALS '{{}}'
            CONFIG '{{"region": "us-east-1"}}'
            INTO PROCEDURE ingest_purchases
            FORMAT PARQUET (
                subscriber_id <- subscriberid,
                vendor <- vendor
            )
        ''')
        t0 = time.perf_counter()
        run_sql('START PIPELINE purchases_pipeline FOREGROUND')
        print(f'Pipeline finished in {time.perf_counter() - t0:.1f}s')
    except Exception as e:
        # Fallback: read a few of the same files over HTTPS and run the same procedure logic.
        import io, urllib.request
        print('Pipeline not available here, loading over HTTPS instead ->', e)
        listing = urllib.request.urlopen(f'{BUCKET_URL}/?list-type=2&prefix=v2/{SCALE}/purchases.').read().decode()
        for key in re.findall(r'<Key>(.*?)</Key>', listing)[:10]:
            df = pd.read_parquet(io.BytesIO(urllib.request.urlopen(f'{BUCKET_URL}/{key}').read()))
            with conn.cursor() as cur:
                cur.executemany('INSERT INTO purchases (subscriber_id, vendor) VALUES (%s, %s)',
                                df[['subscriberid', 'vendor']].values.tolist())
        run_sql('CALL refresh_ads()')


def load_from_demo():
    """Copy up to MAX_CREATIVES offers and recent purchases from the demo's database, then build the ads.

    Copying no more offers than build_creatives() turns into ads keeps offers and ads in step,
    so no leftover offers sneak into the Chapter 6 waves."""
    try:
        run_sql(f'''
            INSERT IGNORE INTO offers
            SELECT offer_id, customer, enabled, notification_zone, segment_ids,
                   notification_content, notification_target, maximum_bid_cents
            FROM {DEMO_DB}.offers
            ORDER BY offer_id LIMIT {MAX_CREATIVES}
        ''')
    except Exception:
        # Older demo versions store the zone as GEOGRAPHY; we don't need it, so skip it.
        run_sql(f'''
            INSERT IGNORE INTO offers
            SELECT offer_id, customer, enabled, NULL, segment_ids,
                   notification_content, notification_target, maximum_bid_cents
            FROM {DEMO_DB}.offers
            ORDER BY offer_id LIMIT {MAX_CREATIVES}
        ''')
    run_sql(f'''
        INSERT INTO purchases (subscriber_id, vendor, ts)
        SELECT subscriber_id, vendor, ts FROM {DEMO_DB}.purchases
        ORDER BY ts DESC LIMIT 100000
    ''')
    run_sql('CALL build_creatives()')

found = run_sql('''
    SELECT COUNT(*) AS n FROM information_schema.TABLES
    WHERE TABLE_SCHEMA = %s AND TABLE_NAME IN ('offers', 'purchases')
''', (DEMO_DB,)).iloc[0, 0]
demo_offers = run_sql(f'SELECT COUNT(*) FROM {DEMO_DB}.offers').iloc[0, 0] if found == 2 else 0

if demo_offers > 0:
    print(f'Found the demo database `{DEMO_DB}` with {demo_offers} offers. Using its live data.')
    t0 = time.perf_counter()
    load_from_demo()
    print(f'Loaded in {time.perf_counter() - t0:.1f}s')
else:
    print(f'No demo database `{DEMO_DB}` found (or it has no offers yet). Streaming purchases from S3 instead.')
    load_from_s3()

embed_pending()
run_sql('''
    SELECT (SELECT COUNT(*) FROM purchases)    AS purchases_loaded,
           (SELECT COUNT(*) FROM offers)       AS offers_created,
           (SELECT COUNT(*) FROM ad_creatives) AS ads_created
''')

In [8]:
run_sql('''
    SELECT offer_id, customer, category, headline, body
    FROM ad_creatives
    ORDER BY RAND()
    LIMIT 5
''')

## Chapter 2: Keyword search

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">12:03 PM</span> &nbsp;<b>Morgan (CMO):</b> Find every ad about <b>wireless headphones</b>. Also, I'll probably type it wrong.
</div>

Add a VERSION 2 full-text index to the table that already holds data:

In [9]:
if not IS_SHARED:
    run_sql(f'USE {DB_NAME}')   # stay in the notebook's database, whatever the drop-down says

run_sql('''
    ALTER TABLE ad_creatives
    ADD FULLTEXT USING VERSION 2 ft_ads (customer, headline, body)
    INDEX_OPTIONS '{"analyzer": "standard"}'
''')
run_sql('OPTIMIZE TABLE ad_creatives FLUSH')   # make the newest rows searchable right away
print('Full-text index added.')

<div class="alert alert-block alert-info"><b class="fa fa-solid fa-gears"></b><div>
<p><b>Under the hood:</b> VERSION 2 full-text search runs on Apache Lucene, the Java search library Elasticsearch is also built on, inside a companion process SingleStore calls the JLucene service. Lucene builds one index per data segment, which matters in Chapter 6.</p>
</div></div>

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>SHOW FULLTEXT INDEX STATUS</b><br/>See how far index building has progressed on each partition, and whether any part is broken (fix with <tt>OPTIMIZE TABLE ... FIX_FULLTEXT</tt>).
</div>

In [10]:
def fts_status():
    """One-line summary of SHOW FULLTEXT INDEX STATUS. Returns the lowest build progress (%)."""
    df = try_sql('SHOW FULLTEXT INDEX STATUS FOR ad_creatives', label='SHOW FULLTEXT INDEX STATUS')
    if df is None or df.empty:
        return None
    progress, broken = pd.to_numeric(df['Progress (%)']), pd.to_numeric(df['Broken (%)'])
    print(f"Full-text index: {progress.min():.0f}% built on all {df['Partition'].nunique()} partitions, "
          f"{broken.max():.0f}% broken")
    return progress.min()

fts_status()

Search with a phrase, then with Morgan's typo (`~1` allows one wrong letter). `BM25` scores each match:

In [11]:
q = 'body:"wireless headphones" OR headline:"wireless headphones"'
run_sql('''
    SELECT customer, category, headline, ROUND(BM25(ad_creatives, %(q)s), 3) AS bm25
    FROM ad_creatives
    WHERE BM25(ad_creatives, %(q)s)
    ORDER BY bm25 DESC
    LIMIT 8
''', {'q': q})

In [12]:
run_sql('''
    SELECT customer, headline
    FROM ad_creatives
    WHERE MATCH (TABLE ad_creatives) AGAINST ('body:hedphones~1')
    LIMIT 5
''')

## Chapter 3: "run" should find "running"

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">2:17 PM</span> &nbsp;<b>Morgan (CMO):</b> I searched <b>run</b> and got nothing from the running shoe people. Fix it.
</div>

In [13]:
run_sql('''
    SELECT COUNT(*) AS ads_matching_run
    FROM ad_creatives
    WHERE MATCH (TABLE ad_creatives) AGAINST ('body:run')
''')

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>ANALYZE FULLTEXT</b><br/>Shows the exact words (tokens) an analyzer produces from any text, without touching a table. Perfect for debugging search.
</div>

In [14]:
phrase = 'Running shoes for runners who ran the marathon'

def tokens(options=None):
    """Return the tokens ANALYZE FULLTEXT produces for `phrase` (one row per token)."""
    q = 'ANALYZE FULLTEXT %s' + (' OPTIONS %s' if options else '')
    df = try_sql(q, (phrase, options) if options else (phrase,), label='ANALYZE FULLTEXT')
    if df is None or df.empty:
        return None
    if 'token' in df.columns:
        return df['token'].tolist()
    # Some versions return a single JSON document instead
    return [t['token'] for t in json.loads(df.iloc[0, 0])['tokens']]

print('standard:', tokens())
print('english: ', tokens('{"analyzer": "english"}'))

The `english` analyzer stems words, so "Running" becomes "run". Swap it in, keeping brand names on `standard`, and watch the rebuild with `SHOW FULLTEXT INDEX STATUS`:

In [15]:
run_sql('DROP INDEX ft_ads ON ad_creatives')
run_sql('''
    ALTER TABLE ad_creatives
    ADD FULLTEXT USING VERSION 2 ft_ads (customer, headline, body)
    INDEX_OPTIONS '{"analyzer": "english", "mappings": {"customer": {"analyzer": "standard"}}}'
''')
run_sql('OPTIMIZE TABLE ad_creatives FLUSH')

for _ in range(15):   # wait for the rebuild to finish
    done = fts_status()
    if done is None or done >= 100:
        break
    time.sleep(2)

In [16]:
run_sql('''
    SELECT customer, category, headline
    FROM ad_creatives
    WHERE MATCH (TABLE ad_creatives) AGAINST ('body:run')
    LIMIT 5
''')

## Chapter 4: Ads that feel like a rainy Sunday

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">4:44 PM</span> &nbsp;<b>Morgan (CMO):</b> People search for FEELINGS. Show me ads that feel like <b>a rainy Sunday</b>.
</div>

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>F16 vectors</b><br/>Half-precision vectors use about half the storage of F32 and, per SingleStore's docs, run exact kNN search about 35% faster with similar recall. Your ads already have them (or F32, if F16 isn't available yet).
</div>

Add an HNSW vector index and search by meaning:

In [17]:
try_sql('''
    ALTER TABLE ad_creatives
    ADD VECTOR INDEX ivx (embedding)
    INDEX_OPTIONS '{"index_type": "HNSW_FLAT", "metric_type": "DOT_PRODUCT"}'
''', label='Vector index')

def set_query_vector(text):
    """Embed text into the session variable @qv, using the same model as the ads."""
    if EMBED_MODE == 'ai':
        run_sql(f'SET @qv = cluster.EMBED_TEXT(%s) :> VECTOR({DIM}) :> {VEC_TYPE}', (text,))
    else:
        v = local_model.encode([text], normalize_embeddings=True)[0]
        run_sql(f'SET @qv = %s :> {VEC_TYPE}', (json.dumps([float(x) for x in v]),))

def vibe_search(text, k=8):
    set_query_vector(text)
    return run_sql(f'''
        SELECT customer, category, headline, ROUND(embedding <*> @qv, 3) AS similarity
        FROM ad_creatives
        ORDER BY similarity DESC
        LIMIT {k}
    ''')

vibe_search('a rainy Sunday')

In [18]:
for mood in ['getting ready for a romantic date night', 'the car is packed and the playlist is ready', 'I need to fix something this weekend']:
    print('Mood:', mood)
    display(vibe_search(mood, k=4))

<div class="alert alert-block alert-info"><b class="fa fa-solid fa-flask"></b><div>
<p><b>Evaluator's corner: F16 vs F32.</b> The next two cells compare storage, exact kNN speed and top-10 agreement on your own workspace.</p>
</div></div>

In [19]:
REPEAT = 20   # copies of each ad; raise for bigger tests

for t in ['bench_f16', 'bench_f32']:
    run_sql(f'DROP TABLE IF EXISTS {t}')
run_sql(f'CREATE TABLE bench_f16 (id BIGINT, embedding {VEC_TYPE}, SHARD KEY (id))')
run_sql(f'CREATE TABLE bench_f32 (id BIGINT, embedding VECTOR({DIM}), SHARD KEY (id))')

for r in range(REPEAT):
    run_sql(f'INSERT INTO bench_f16 SELECT offer_id * 1000 + {r}, embedding FROM ad_creatives WHERE embedding IS NOT NULL')
run_sql(f'INSERT INTO bench_f32 SELECT id, embedding :> VECTOR({DIM}) FROM bench_f16')
run_sql('OPTIMIZE TABLE bench_f16 FULL')
run_sql('OPTIMIZE TABLE bench_f32 FULL')

sizes = try_sql('''
    SELECT TABLE_NAME AS table_name,
           SUM(ROWS_COUNT) AS rows_stored,
           ROUND(SUM(COMPRESSED_SIZE) / 1024 / 1024, 2) AS embedding_mb_on_disk
    FROM information_schema.COLUMNAR_SEGMENTS
    WHERE DATABASE_NAME = DATABASE() AND TABLE_NAME IN ('bench_f16', 'bench_f32') AND COLUMN_NAME = 'embedding'
    GROUP BY TABLE_NAME
''', label='Storage comparison')
sizes

In [20]:
def median_ms(query, runs=7):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        run_sql(query)
        times.append((time.perf_counter() - t0) * 1000)
    return round(statistics.median(times), 1)

moods = ['a rainy Sunday', 'date night', 'road trip', 'home workout', 'birthday gift for a kid']
overlap, t16, t32 = [], [], []
for mood in moods:
    set_query_vector(mood)
    run_sql(f'SET @qv32 = @qv :> VECTOR({DIM})')
    # speed: exact kNN over every row (no vector index on the bench tables)
    t16.append(median_ms('SELECT id FROM bench_f16 ORDER BY embedding <*> @qv DESC LIMIT 10'))
    t32.append(median_ms('SELECT id FROM bench_f32 ORDER BY embedding <*> @qv32 DESC LIMIT 10'))
    # quality: do F16 and F32 pick the same 10 distinct ads? (each ad is repeated REPEAT times)
    top = lambda t, v: set(run_sql(f'SELECT id DIV 1000 AS ad, MAX(embedding <*> {v}) AS s FROM {t} '
                               f'GROUP BY ad ORDER BY s DESC LIMIT 10')['ad'])
    overlap.append(len(top('bench_f16', '@qv') & top('bench_f32', '@qv32')) / 10)

if VEC_ELEM != 'F16':
    print('Note: F16 is not available on this workspace, so both tables below use F32. '
          'Rerun on a workspace with F16 enabled to see the real difference.')
pd.DataFrame({
    'metric': ['median exact kNN latency (ms)', 'top-10 agreement with F32'],
    f'{VEC_ELEM} (bench_f16)': [statistics.median(t16), f'{statistics.mean(overlap):.0%}'],
    'F32 (bench_f32)': [statistics.median(t32), '100% (baseline)'],
})

## Chapter 5: Hybrid search

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">6:10 PM</span> &nbsp;<b>Morgan (CMO):</b> I want <b>cozy headphones for a rainy day</b>. Keywords AND feelings. One answer.
</div>

Run both searches and blend them with Reciprocal Rank Fusion in one SQL query. `found_by` shows which search found each ad:

In [21]:
def lucene(text):
    """Turn free text into a safe Lucene query over the indexed columns."""
    words = ' '.join(re.findall(r'[A-Za-z0-9]+', text))
    return f'headline:({words}) OR body:({words})'

HYBRID_SQL = '''
WITH kw AS (
    SELECT offer_id, BM25(ad_creatives, %(lq)s) AS s
    FROM ad_creatives
    WHERE BM25(ad_creatives, %(lq)s)
    ORDER BY s DESC LIMIT 50
),
vec AS (
    SELECT offer_id, embedding <*> @qv AS s
    FROM ad_creatives
    ORDER BY s DESC LIMIT 50
),
ranked AS (
    SELECT offer_id, ROW_NUMBER() OVER (ORDER BY s DESC) AS r, 'keyword' AS src FROM kw
    UNION ALL
    SELECT offer_id, ROW_NUMBER() OVER (ORDER BY s DESC) AS r, 'vector' AS src FROM vec
),
fused AS (
    SELECT offer_id,
           SUM(1.0 / (60 + r)) AS rrf,
           GROUP_CONCAT(CONCAT(src, ' #', r) SEPARATOR ', ') AS found_by
    FROM ranked GROUP BY offer_id
)
SELECT a.customer, a.category, a.headline, a.max_bid_cents, ROUND(f.rrf, 4) AS rrf, f.found_by
FROM fused f JOIN ad_creatives a ON a.offer_id = f.offer_id
ORDER BY f.rrf DESC
LIMIT 10
'''

def hybrid_search(text):
    set_query_vector(text)
    return run_sql(HYBRID_SQL, {'lq': lucene(text)})

hybrid_search('cozy headphones for a rainy day')

Add business rules with a plain `WHERE`, such as bids of at least 10 cents:

In [22]:
set_query_vector('cozy headphones for a rainy day')
run_sql(HYBRID_SQL.replace('ORDER BY f.rrf DESC', 'WHERE a.max_bid_cents >= 10\nORDER BY f.rrf DESC'),
    {'lq': lucene('cozy headphones for a rainy day')})

## Chapter 6: Super Bowl night

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">7:58 PM</span> &nbsp;<b>Morgan (CMO):</b> 10x the ads tonight. If search slows down, I'm replacing you with a spreadsheet.
</div>

New data lands in small segments, and each segment gets its own vector index and full-text index. Many small indexes mean more work per search. **Index merging** combines them, but vector and full-text merging are separate mechanisms:

| | Vector index merging | Full-text index merging |
| --- | --- | --- |
| Merges | HNSW/IVF structures | Lucene inverted indexes |
| Runs in | The SingleStore engine | The JLucene service |
| Setting | `background_vector_index_merger` | `enable_fts2_merger` |
| Trigger | Background, or `OPTIMIZE TABLE ... VECTOR_INDEX` | Background only |
| Observe | Segment counts, latency | `MV_FTS_MERGED_INDEXES` |

Stream in 12 waves of new ads, each flushed into its own segment:

In [23]:
def index_segments():
    """Each columnstore segment carries its own vector and full-text index,
    so counting segments shows how many small indexes a query has to probe."""
    df = try_sql('''
        SELECT COUNT(DISTINCT PARTITION, SEGMENT_ID) AS segments,
               SUM(ROWS_COUNT - DELETED_ROWS_COUNT) AS live_rows
        FROM information_schema.COLUMNAR_SEGMENTS
        WHERE DATABASE_NAME = DATABASE() AND TABLE_NAME = 'ad_creatives' AND COLUMN_NAME = 'offer_id'
    ''', label='Segment counts')
    if df is not None:
        display(df)

print('Before kickoff:')
index_segments()

In [24]:
WAVES, ADS_PER_WAVE = 12, 100   # 1,200 new ads; lower these to save AI Function quota

templates = run_sql('SELECT notification_zone, segment_ids FROM offers LIMIT 1').iloc[0]
next_id = int(run_sql('SELECT MAX(offer_id) AS m FROM offers').iloc[0, 0]) + 1
vendor_rows = vendors[['vendor', 'domain']].values.tolist()

for wave in range(1, WAVES + 1):
    rows = []
    for _ in range(ADS_PER_WAVE):
        vendor, domain = random.choice(vendor_rows)
        rows.append((next_id, vendor, True, templates['notification_zone'], json.dumps([]),
                     f'{random.randint(10, 70)}% off at {vendor} during the big game', domain,
                     random.randint(5, 40)))
        next_id += 1
    with conn.cursor() as cur:
        cur.executemany('INSERT INTO offers VALUES (%s, %s, %s, %s, %s, %s, %s, %s)', rows)
    run_sql('CALL build_creatives()')
    embed_pending()
    run_sql('OPTIMIZE TABLE ad_creatives FLUSH')   # each wave becomes its own segment
    print(f'Wave {wave}: {ADS_PER_WAVE} ads live')

print('\nAfter the rush:')
index_segments()

In [25]:
QUERY = 'snacks and a big screen for the game'
set_query_vector(QUERY)
LQ = lucene(QUERY)

VECTOR_ONLY = 'SELECT offer_id FROM ad_creatives ORDER BY embedding <*> @qv DESC LIMIT 10'
KEYWORD_ONLY = '''
    SELECT offer_id, customer, BM25(ad_creatives, %(lq)s) AS s
    FROM ad_creatives WHERE BM25(ad_creatives, %(lq)s)
    ORDER BY s DESC LIMIT 10
'''

def latency_ms(query, params=None, runs=9):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        run_sql(query, params)
        times.append((time.perf_counter() - t0) * 1000)
    return round(statistics.median(times), 1)

results = {'vector search (ms)': [latency_ms(VECTOR_ONLY)],
           'keyword search (ms)': [latency_ms(KEYWORD_ONLY, {'lq': LQ})]}
top_keyword_before = run_sql(KEYWORD_ONLY, {'lq': LQ})
pd.DataFrame(results, index=['after the rush'])

### Vector index merging

In [26]:
try_sql('OPTIMIZE TABLE ad_creatives VECTOR_INDEX ivx', label='Vector index merge')
results['vector search (ms)'].append(latency_ms(VECTOR_ONLY))
print(f"Vector search: {results['vector search (ms)'][0]} ms -> {results['vector search (ms)'][1]} ms")

### Full-text index merging

<div style="border: 2px solid #d199ff; border-radius: 6px; padding: 8px 14px; margin: 6px 0;">
<span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> &nbsp;<b>Full-text index merger (rolling out)</b><br/>Combines per-segment Lucene indexes into cross-segment indexes, with compaction, in the background. It may not be active on your workspace yet; the next cell shows its setting and anything it has merged.
</div>

In [27]:
settings = run_sql("SHOW VARIABLES WHERE Variable_name IN ('enable_fts2_merger', 'background_vector_index_merger')")
display(settings)

merged = try_sql('''
    SELECT PARTITION, FTS_INDEX_ID, NUM_SEGMENTS, NUM_ROWS, SIZE
    FROM information_schema.MV_FTS_MERGED_INDEXES
    WHERE DATABASE_NAME = DATABASE() AND TABLE_NAME = 'ad_creatives'
    ORDER BY PARTITION
''', label='MV_FTS_MERGED_INDEXES')

if merged is not None and len(merged) > 0:
    print(f'The full-text merger combined per-segment indexes into {len(merged)} merged index(es):')
    display(merged)
else:
    print('No merged full-text indexes on this workspace yet. The full-text index merger is new and still rolling out; '
          'once it is active, this view fills in automatically as new segments arrive. Nothing to change in the notebook.')

fts_status()

results['keyword search (ms)'].append(latency_ms(KEYWORD_ONLY, {'lq': LQ}))
print(f"Keyword search: {results['keyword search (ms)'][0]} ms -> {results['keyword search (ms)'][1]} ms")
print('(Background merges of data segments and warm caches also affect this number, so compare several runs.)')

Merging should never change results. Check that the top keyword matches are identical, then see the scoreboard:

In [28]:
top_keyword_after = run_sql(KEYWORD_ONLY, {'lq': LQ})
pd.DataFrame({
    'before: advertiser': top_keyword_before['customer'].values,
    'before: bm25': top_keyword_before['s'].round(3).values,
    'after: advertiser': top_keyword_after['customer'].values,
    'after: bm25': top_keyword_after['s'].round(3).values,
})

In [29]:
try_sql('OPTIMIZE TABLE ad_creatives', label='Columnstore merge')
index_segments()
pd.DataFrame(results, index=['after the rush', 'after merging'])

<div class="alert alert-block alert-info"><b class="fa fa-solid fa-flask"></b><div>
<p><b>Evaluator's corner: merging at scale.</b> With a few thousand rows every index is tiny, so differences are small. Raise <tt>SCALE</tt> and <tt>WAVES</tt> to test under realistic write loads, and compare several runs, since caches and background merges also affect timings.</p>
</div></div>

<div style="border-left: 4px solid #d199ff; background: rgba(209,153,255,0.10); padding: 8px 14px;">
<b>#war-room</b> <span style="opacity:0.7">11:59 PM</span> &nbsp;<b>Morgan (CMO):</b> Search never blinked. You're a genius. Tomorrow: Spanish. (Easy: swap in the <tt>spanish</tt> analyzer, as in Chapter 3.)
</div>

## What you built

- **F16 vectors** <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> for smaller, faster embeddings, searched by meaning
- **Full-text search VERSION 2** with fuzzy matching and BM25, added to a live table
- **Full-text manageability** <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span>: `ANALYZE FULLTEXT`, `SHOW FULLTEXT INDEX STATUS`, per-column analyzers
- **Hybrid search** in one SQL query
- **Index merging**: vector index merging, plus the <span style="background:#d199ff; color:#1b1b1b; font-weight:700; border-radius:4px; padding:2px 8px; font-size:85%;">NEW</span> full-text index merger

## Clean up

Drops the `adventure_inc` database on Standard and Premium workspaces (the demo's `martech` database is left alone).

In [30]:
try_sql('DROP PIPELINE IF EXISTS purchases_pipeline', label='Drop pipeline')
if not IS_SHARED:
    run_sql(f'DROP DATABASE IF EXISTS {DB_NAME}')
else:
    for t in ['bench_f16', 'bench_f32', 'ad_creatives', 'creatives_staging', 'offers', 'purchases', 'category_vibes', 'vendors']:
        run_sql(f'DROP TABLE IF EXISTS {t}')
    for p in ['ingest_purchases', 'refresh_ads', 'build_creatives']:
        try_sql(f'DROP PROCEDURE IF EXISTS {p}', label=f'Drop {p}')
print('All clean. Morgan will never know.')

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>